# **Modeling**

In [1]:
import pandas as pd
import numpy as np
import joblib

train_df = pd.read_csv("train_processed.csv")
test_df = pd.read_csv("test_processed.csv")

train_df.head()

,Area,Perimeter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,Class,Class_encoded
0,0.231296,-0.038760,-0.273192,0.945150,-0.946017,0.237163,0.909727,Osmancik,1
1,1.564176,1.378486,1.320193,1.235297,0.537474,1.569790,1.007847,Cammeo,0
2,-0.809311,-0.960188,-1.033130,-0.164148,-1.017535,-0.779219,-0.249349,Osmancik,1
3,-0.156472,0.000461,0.178910,-0.605509,0.716758,-0.169277,-1.016331,Osmancik,1
4,0.840728,1.017703,1.037150,0.335035,0.855321,0.876418,1.170966,Cammeo,0


In [2]:
#Prepare X / y
feature_cols = [c for c in train_df.columns if c not in ["Class", "Class_encoded"]]

X_train = train_df[feature_cols]
y_train = train_df["Class_encoded"]

X_test = test_df[feature_cols]
y_test = test_df["Class_encoded"]

print("Features used:", feature_cols)
print("X_train:", X_train.shape, " X_test:", X_test.shape)

Features used: ['Area', 'Perimeter', 'Major_Axis_Length', 'Minor_Axis_Length', 'Eccentricity', 'Convex_Area', 'Extent']
X_train: (3048, 7)  X_test: (762, 7)


---
## Model 1 — Baseline: Logistic Regression


In [3]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)

train_acc = log_reg.score(X_train, y_train)
test_acc = log_reg.score(X_test, y_test)

print(f"Logistic Regression — Train accuracy: {train_acc:.4f} | Test accuracy: {test_acc:.4f}")

Logistic Regression — Train accuracy: 0.9321 | Test accuracy: 0.9173


## Model 2 — Decision Tree

In [4]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42, max_depth=6)
dt.fit(X_train, y_train)

train_acc = dt.score(X_train, y_train)
test_acc = dt.score(X_test, y_test)

print(f"Decision Tree — Train accuracy: {train_acc:.4f} | Test accuracy: {test_acc:.4f}")

Decision Tree — Train accuracy: 0.9446 | Test accuracy: 0.9121


## Model 3 — Random Forest

In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=None)
rf.fit(X_train, y_train)

train_acc = rf.score(X_train, y_train)
test_acc = rf.score(X_test, y_test)

print(f"Random Forest — Train accuracy: {train_acc:.4f} | Test accuracy: {test_acc:.4f}")

Random Forest — Train accuracy: 1.0000 | Test accuracy: 0.9160


## Model 4 — SVM *or* KNN

In [6]:
from sklearn.svm import SVC

svm = SVC(kernel="rbf", C=1.0, random_state=42)
svm.fit(X_train, y_train)

train_acc = svm.score(X_train, y_train)
test_acc = svm.score(X_test, y_test)

print(f"SVM (RBF) — Train accuracy: {train_acc:.4f} | Test accuracy: {test_acc:.4f}")

SVM (RBF) — Train accuracy: 0.9350 | Test accuracy: 0.9108


In [7]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

train_acc = knn.score(X_train, y_train)
test_acc = knn.score(X_test, y_test)

print(f"KNN (k=5) — Train accuracy: {train_acc:.4f} | Test accuracy: {test_acc:.4f}")

KNN (k=5) — Train accuracy: 0.9373 | Test accuracy: 0.9094


## Quick side-by-side


In [8]:
results = {
    "Logistic Regression": log_reg.score(X_test, y_test),
    "Decision Tree": dt.score(X_test, y_test),
    "Random Forest": rf.score(X_test, y_test),
    "SVM": svm.score(X_test, y_test),
    "KNN": knn.score(X_test, y_test),
}

results_df = pd.DataFrame.from_dict(results, orient="index", columns=["Test Accuracy"]).sort_values("Test Accuracy", ascending=False)
results_df

,Test Accuracy
Logistic Regression,0.917323
Random Forest,0.916010
Decision Tree,0.912073
SVM,0.910761
KNN,0.909449


## Save trained models (Evaluation)


In [9]:
joblib.dump(log_reg, "model_logistic_regression.pkl")
joblib.dump(dt, "model_decision_tree.pkl")
joblib.dump(rf, "model_random_forest.pkl")
joblib.dump(svm, "model_svm.pkl")
joblib.dump(knn, "model_knn.pkl")

print("Saved 5 trained model files for the evaluation step.")

Saved 5 trained model files for the evaluation step.


## Modeling Decision Log

**Data used:** `train_processed.csv` / `test_processed.csv` exactly as produced by Person 2 — no re-scaling, re-splitting, or re-encoding done here, to keep results comparable across the group.

**Target:** `Class_encoded` (numeric) used for all five models, since sklearn's SVM/KNN/tree implementations expect numeric labels; the string `Class` column is kept in the CSVs only for readability.

**Baseline — Logistic Regression:** `max_iter=1000` raised from the default (100) because the model didn't converge at default settings on this feature set; all other parameters left at sklearn defaults, since a baseline should be the simplest reasonable version, not a tuned one.

**Decision Tree:** `max_depth=6` set to keep the tree from overfitting to noise/outlier caps, while still deep enough to capture the multicollinear size features (Area, Perimeter, Convex_Area) meaningfully. `random_state=42` for reproducibility, matching Person 2's split.

**Random Forest:** `n_estimators=200` for a stable average across trees; `max_depth=None` left unrestricted since the ensemble itself controls overfitting through averaging — this also gives a cleaner feature-importance signal for the group's secondary lens (feature-importance analysis).

**SVM vs KNN:** Both trained since the workflow allows either. SVM (RBF kernel, default `C=1.0`) generally handles the correlated size features better since it isn't distance-based in the raw feature space; KNN (`k=5`, an odd-ish default to reduce tie risk on a binary target) was kept as a fallback in case the group prefers a simpler, non-parametric baseline for the model comparison. Recommend the group picks based on Person 4's evaluation numbers rather than assuming SVM wins.

**Reproducibility:** `random_state=42` used everywhere a model exposes it, matching the seed used in Person 2's train/test split, so results are reproducible end-to-end.

**Handoff to evaluation (Person 4):** All five fitted models saved as `.pkl` files (`model_logistic_regression.pkl`, `model_decision_tree.pkl`, `model_random_forest.pkl`, `model_svm.pkl`, `model_knn.pkl`) alongside `test_processed.csv`, so Person 4 can load each model directly and compute accuracy, precision, recall, F1, and confusion matrices without retraining.